# Part One: Multi-armed Bandits

## Section 1.1: Project Overview 

Part One investigates the trade-off between exploration and exploitation using the Multi-armed Bandit problem to better understand the fundamental objective of reinforcement learning; that is, maximizing reward through empirical interaction with an uncertain environment, under limited information and with finite resources. In theory, if one had perfect and complete information about the value of each action, a strategy of pure exploitation would maximize the expected total reward. However, under conditions of uncertainty, exploration can produce the greater reward by discovering actions with higher value (Sutton & Barto, Chapter 2). 

I implement and compare two strategies for balancing exploration with exploitation:

- **ε-greedy:** a strategy that selects actions randomly a fraction of the time, but otherwise takes the action with the highest value estimate

- **Upper Confidence Bound (UCB):** a strategy that selects the action with the highest value estimate, adjusted by a bonus that favors less-sampled actions, achieving exploration without random selection.

The comparison is carried out by agents configured with one or the other of these two strategies. It takes place in an action space of k arms, which we will initialize to 10 to match the 10-armed testbed from Sutton & Barto, Chapter 2. Each arm's true mean is sampled from N(0,1) at the start of each run, and rewards are drawn from a Gaussian distribution centered on that value. Each run ends after 2,000 steps, and results for each agent are averaged over 1,000 runs.

The resulting comparison is expected to demonstrate that the UCB strategy yields the best results in the long run, because it more efficiently manages exploration by targeting those actions about which we know the least, and by so doing, improves the confidence in long-term estimates of value. By contrast, the random approach of the ε-greedy algorithm wastes effort by indiscriminately retrying actions about which enough information to estimate long-term value has already been collected, and so takes longer to learn to optimize actions and maximize reward.

## Section 1.2: Deliverables

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W1

I implemented a Python framework for running multi-armed bandit simulations using Gymnasium. It follows the Gymnasium API (`reset()` / `step()`) and supports randomized arm distributions across independent runs, matching the 10-armed testbed from Sutton & Barto, Chapter 2.

### Implementation Summary

The framework consists of the following classes:

- **MultiArmedBanditEnv:** A k-armed bandit environment following the Gymnasium API.

- **BanditAgent:** Abstract base class. Concrete implementations:

    - **EpsilonGreedyAgent:** Selects the best-known lever most of the time, but explores randomly with probability ε.

    - **UCBAgent:** Selects levers based on observed reward plus a confidence bonus that favors underexplored options.

- **BanditSimulator:** Executes multiple independent episodes, producing a SimulationResult.

- **SimulationResult:** Collects raw results and provides averaged metrics (reward, optimal action %, cumulative regret).

- **BanditPlotter:** Plots average reward, optimal action percentage, and cumulative regret over time.

- **BanditFactory:** Constructs environments and agents by name.

A 10-arm bandit environment runs 1,000 episodes of 2,000 steps each per agent. Five agents are compared: three ε-greedy (ε = 0.01, 0.1, 0.2) and two UCB (c = 0.1, 0.2). The environment resets each episode with fresh reward distributions, ensuring independence.

### Key Results and Analysis

Each agent performed 1,000 runs of 2,000 steps. Two metrics were collected:

- **Average reward over time:** measures how much reward an agent earns per step, averaged across all runs. This shows how quickly an agent learns to identify and exploit high-value arms. An agent that explores efficiently will see this curve rise steeply and plateau near the true mean of the best arm.

- **Optimal action percentage:** measures how often the agent selects the single best arm at each time step, expressed as a percentage across all 1,000 runs. A high optimal action percentage indicates that the agent has correctly identified which arm has the highest expected value and consistently chooses it.

#### Average Reward

Figure 1.1 shows the average reward of the five agents over 2,000 steps. The UCB agents consistently outperformed the ε-greedy agents, accumulating more total reward across the evaluation period.

<figure>
    <img src="../part_01/" alt="Average reward over time" width='100%'>
    <figcaption>Figure 1.1: Comparison of average reward per time-step for five agents over 2,000 steps</figcaption>
</figure>

The ε-greedy agents gain an early advantage by quickly exploiting a promising lever, while the UCB agents methodically sample all options first. This lead is short-lived. Among the ε-greedy agents, higher ε values identify the optimal lever sooner, but they continue to randomly explore even after they have enough information to estimate which lever is the best. As a consequence, they are wasting pulls on suboptimal levers which is costing them reward roughly proportional to the epsilon value. Meanwhile, ε=0.01 explores so rarely that it is still improving at step 2,000, suggesting it has not yet fully converged. By exploiting almost exclusively, it wastes fewer pulls and achieves the highest average reward among the ε-greedy agents.

#### Optimal Action Percentage

Figure 1.2 shows the optimal action percentage of the five agents over 2,000 steps. Here, too, we see that the UCB agents outperform the ε-greedy agents.

<figure>
    <img src="../part_01/img/bandit_optimal_action.png" alt="Optimal action percentage over time" width='100%'>
    <figcaption>Figure 1.1: Comparison of optimal action percentage per time-step for five agents over 2,000 steps</figcaption>
</figure>

The chart reflects the learning rate of the agents as they gathered information to decide which actions would yield the highest reward. The UCB agents learned the fastest, and we can clearly see that a higher exploration weight dampens the learning rate, provoking the model into more exploration that might be necessary.  Among the ε-greedy agents, we can see two dynamics at play. First, higher values of ε speed up learning, but there is a tipping point at which the ε value imposes a cost, a threshold beyond which the agent can't optimize. In the ε-greedy agent with the smallest ε value of 0.01, learning is so slow that the agent hasn't yet found that threshold, but we can estimate that it would plateau near 99%, eventually outperforming the UCB agents, given enough time.

The experiment confirms the observations found in Sutton and Barto, Chapter 2.7.

## Section 1.3: AI Use Reflection

### Initial Interaction

My initial prompt to Claude was `Let's walk through the multi-arm bandit problem together`. 

Claude explained the setup: an agent faces *k* slot machines with unknown payout probabilities and must maximize total reward. The core tension is exploration versus exploitation. Claude then outlined three strategies (ε-greedy, UCB, and Thompson Sampling), describing how each balances trying new arms against leveraging known information.

### Iteration Cycle

After the initial walkthrough, I said I wanted to design a Python framework for bandit simulations using Gymnasium. Claude jumped ahead and generated a full codebase immediately. I pushed back, asking Claude to discuss the design before writing code. Claude pivoted to a consumer-first approach; we started with pseudocode showing how someone would use the framework, then worked through configuration choices (simple dicts, factory functions, readable result wrappers) before writing any implementation.

We reviewed the design against the lab requirements: 10 Gaussian arms, ε-greedy and UCB agents, 2,000 steps across 1,000 runs, and comparison plots with multiple parameter settings. Claude checked each requirement against the proposed interfaces and surfaced two open questions: whether plot panels should be configurable, and whether arm means should be randomized per run or fixed. I chose configurable panels and randomized means.

I asked Claude to render a UML class diagram before any code was written. Claude produced an inline SVG showing the class hierarchy, including four concrete agents, the simulation runner, result object, and factory functions. I asked to drop Thompson Sampling and Boltzmann, and Claude re-rendered with only EpsilonGreedy and UCB. I also asked about exporting the diagram; Claude produced a Mermaid version, which we inlined into the README.

### Critical Evaluation

The iterative, consumer-first design process ensured clear requirements and an implementation organized for easy inspection, allowing us to iterate on design and clarify class responsibilities. Comparing the final implementation to Claude's first attempt shows how much this process mattered: the initial codebase lacked factories, randomization, and configurable plotting. The final version addressed all three, scoped the agents to lab requirements, and produced a meaningfully different API.

### Learning Reflection

I learned that Claude will eagerly generate code without boundaries, so I established a pattern early: propose first, review, give consent, then implement. I used Claude as a design partner, not just a code generator, insisting on interface-first design and UML diagrams before implementation. I also learned to verify Claude's claims against evidence; when Claude reviewed the research summary, it mischaracterized the plots. I caught it and pushed back. Claude's first output isn't necessarily its best, and iterative refinement consistently improved the result.

## Section 1.4: Speaker Notes

- **Problem:** Exploration vs. exploitation in a 10-armed bandit with stationary Gaussian rewards, investigating how different strategies balance discovering high-value arms against maximizing cumulative reward.

- **Method:** Implemented ε-greedy (ε = 0.01, 0.1, 0.2) and UCB (c = 0.1, 0.2) agents using a custom Gymnasium-based framework; each agent ran 1,000 episodes of 2,000 steps with freshly randomized arm means per episode.

- **Design choice:** Adopted a consumer-first, interface-driven design process, starting with pseudocode and UML diagrams before writing any implementation. This produced a modular framework with factories, configurable plotting, and clean separation of environment, agent, simulation, and results.

- **Key result:** UCB agents consistently outperformed ε-greedy agents in both average reward and optimal action percentage, confirming the findings in Sutton & Barto, Chapter 2.7.

- **Insight:** Higher ε speeds up early learning but imposes a permanent ceiling on performance, while UCB's principled exploration converges faster without that ceiling. The ε=0.01 agent was still improving at step 2,000, suggesting it would eventually reach ~99% optimal action rate given enough time.

- **Challenge:** Claude eagerly generated a full codebase on the first prompt. I pushed back and established a pattern of propose-review-consent-implement, which led to a significantly better API than the initial attempt.

- **Connection:** The exploration-exploitation tradeoff studied here is foundational to full MDP problems in Week 2, where the same tension applies across states and longer time horizons.

# Part Two: Gymnasium API

## Section 2.1: Project Overview

Part Two is an exploration of Gymnasium environments. The purpose of this part of the lab is to become familiar with the Gymnasium API and the constructs that it implements. 

It explores two environments: 

- **FrozenLake-v1:** cross a frozen lake from start to goal without falling into any holes; actions may not always result in the intended outcome due to the slippery nature of the frozen lake.
- **Taxi-v3:** move to the passenger’s location, pick up the passenger, move to the passenger’s desired destination, and drop off the passenger.

I will demonstrate the use of the Env API to introspect attributes of the envrionment, specifically:

- **action_space:** a Space object that corresponds to all actions available in the environment
- **observation_space:** a Space object that defines what an Agent can see

Agents select actions from the action space, the environment processes those actions, and returns an observation from the observation space, along with a reward, termination and truncation flags, and an information dict. 

The lab will demonstrate how an Agent performs these actions, and how we can measure its performance.

## Section 2.2: Deliverables

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W1

I implemented a Python framework for running episodic simulations of agents in standard Gymnasium environments. The framework follows the Gymnasium API (`reset()` / `step()`) and supports collecting per-episode metrics across independent runs, providing a reusable structure for evaluating agent performance.

### Implementation Summary

The framework consists of the following classes and utilities:

- **Agent:** Abstract base class for Gymnasium environment agents. Concrete implementation:

    - **RandomAgent:** Baseline agent that selects actions uniformly at random from the action space.

- **EpisodeResult:** A dataclass that collects per-episode returns, lengths, and success indicators, and provides aggregate summary metrics (mean return, standard deviation, mean length, success rate).

- **run_episodes():** Executes multiple independent episodes of an agent in an environment, collecting per-episode metrics into an EpisodeResult.

- **env_inspector:** Utility module for introspecting Gymnasium environments, including `inspect_env()` for summarizing observation and action spaces, `describe_space()` for structured space descriptions, `get_transition_table()` for extracting transition probabilities from tabular environments, and `format_inspection()` for human-readable output.

- **plot_episode_metrics():** Creates a two-panel visualization showing the distribution of episode returns and the rolling success rate over episodes.


A RandomAgent was evaluated on two environments: FrozenLake-v1 and Taxi-v3, each over 1,000 episodes. The random agent serves as a baseline, establishing the performance floor against which more sophisticated agents can be compared in future work.

### Key Results and Analysis

#### FrozenLake-v1

FrozenLake-v1 is a grid-world environment in which the agent must navigate a 4x4 frozen lake from a start tile to a goal tile without falling into any holes. The observation space is Discrete(16), representing the 16 grid positions. The action space is Discrete(4), corresponding to four movement directions: left, down, right, and up. Critically, because the surface is slippery, the intended action is only executed one-third of the time; the remaining two-thirds of the time, the agent slides perpendicular to the intended direction. This stochasticity makes the environment significantly harder than a deterministic grid-world.

<figure>
    <img src="../part_02/img/frozenlake_random.png" alt="FrozenLake random agent performance" width='100%'>
    <figcaption>Figure 2.1: Random agent performance on FrozenLake-v1 over 1,000 episodes</figcaption>
</figure>

The random agent achieves a low success rate on FrozenLake-v1, which is expected. Without any learning, the agent has no strategy for avoiding holes or navigating toward the goal. The slippery dynamics compound this difficulty, as even an agent that happened to choose good actions would frequently be carried off course. The distribution of episode returns is heavily skewed toward zero, with only occasional successful episodes where the agent stumbled upon the goal by chance. This establishes a clear baseline: any learning agent should substantially outperform random action selection on this task.

#### Taxi-v3

Taxi-v3 is a grid-world environment in which a taxi must navigate to a passenger, pick them up, drive to the destination, and drop them off. The observation space is Discrete(500), encoding the taxi's position, the passenger's location (one of four fixed locations or inside the taxi), and the destination. The action space is Discrete(6): four movement directions plus pickup and dropoff. Illegal pickup or dropoff actions incur a -10 penalty, each movement step costs -1, and successful delivery yields +20.

<figure>
    <img src="../part_02/img/taxi_random.png" alt="Taxi random agent performance" width='100%'>
    <figcaption>Figure 2.3: Random agent performance on Taxi-v3 over 1,000 episodes</figcaption>
</figure>

The random agent performs poorly on Taxi-v3, accumulating large negative returns. The -1 per-step cost and frequent -10 penalties from illegal pickup and dropoff actions produce consistently negative episode returns. The success rate is effectively zero, as randomly selecting from six actions is extremely unlikely to produce the precise sequence required to navigate to the passenger, pick up, navigate to the destination, and drop off. This is a harder coordination problem than FrozenLake, and the baseline performance reflects that. The large negative returns provide a clear floor for evaluating future learning agents.

### Environment Introspection

A key objective of Part Two was to demonstrate the Gymnasium API's introspection capabilities. The `env_inspector` module provides structured access to environment metadata:

- **Observation space:** Describes what the agent can observe. FrozenLake uses a single discrete integer (grid position), while Taxi encodes a compound state (position, passenger location, destination) into a single discrete integer across 500 states.

- **Action space:** Describes what the agent can do. Both environments use discrete action spaces, but Taxi's six actions include two that are context-dependent (pickup and dropoff), introducing the possibility of penalties for illegal actions.

- **Transition table:** For tabular environments, `get_transition_table()` extracts the full P(s'|s,a) dynamics, exposing the MDP structure directly. In FrozenLake, this reveals the stochastic slip probabilities; in Taxi, it shows the deterministic transition and reward structure.

These introspection tools make it possible to verify environment behavior programmatically, which is valuable for debugging agent implementations and for understanding the structure of the MDP before designing a solution.

## Section 2.3: AI Use Reflection

### Initial Interaction

Building on the design-first workflow established in Part One, I approached Part Two with a clearer sense of how to collaborate with Claude. My initial prompt asked Claude to help me understand the Gymnasium API by walking through the key abstractions: environments, spaces, observations, and the step loop. Claude provided a structured overview of the API, covering `reset()`, `step()`, observation and action spaces, and the `info` dict, which gave me a solid mental model before writing any code.

### Iteration Cycle

I reused the propose-review-consent-implement pattern from Part One. We started by identifying the reusable components from the bandit framework (the abstract Agent base class, the simulation runner pattern) and discussed what needed to change for episodic environments with terminal states, as opposed to the fixed-horizon bandit runs.

Claude and I worked through the design of the `env_inspector` module together. I wanted a way to programmatically examine any Gymnasium environment's structure without manually reading documentation. Claude proposed a set of utility functions, and we iterated on what information to extract: space descriptions, reward ranges, max episode steps, and transition tables for tabular environments. I pushed to include `get_transition_table()` after realizing it would let us verify the MDP dynamics directly, which proved valuable for understanding FrozenLake's slip probabilities.

For the runner, we adapted the bandit simulation pattern to handle episodic environments with variable-length episodes, terminal states, and a success criterion. Claude suggested tracking per-episode returns, lengths, and success indicators, which I agreed would provide the metrics needed for meaningful comparison. We also designed the plotting utility to show both the distribution of returns and the rolling success rate, giving two complementary views of agent performance.

### Critical Evaluation

The design-first workflow continued to pay off. By discussing the differences between bandit and episodic environments before writing code, we avoided retrofitting the runner to handle terminal states and variable-length episodes. The `env_inspector` module was a direct result of the iterative process; it started as a simple space printer and grew into a more complete introspection toolkit after we identified the value of exposing transition tables.

One area where Claude's initial output needed correction was in describing the FrozenLake dynamics. Claude initially understated the impact of the slippery mechanic, describing it as occasional randomness rather than the dominant factor it is (the intended action only executes one-third of the time). Verifying this against the transition table confirmed the correct behavior and reinforced the value of programmatic introspection over relying on descriptions alone.

### Learning Reflection

The Gymnasium API exploration deepened my understanding of how reinforcement learning environments are structured. Working with the `env_inspector` module made the relationship between observation spaces, action spaces, and transition dynamics concrete rather than abstract. Seeing FrozenLake's transition table directly revealed why the environment is so challenging for random agents, and why learning algorithms that account for stochastic transitions are essential.

The random agent baseline, while simple, was a valuable exercise. It established a performance floor that makes the value of learning algorithms immediately visible: any agent that performs worse than random selection has a bug, and any agent that performs significantly better is demonstrably learning. This framing will carry forward into Week 2, where policy iteration and value iteration should dramatically outperform random action selection on these same environments.

## Section 2.4: Speaker Notes

- **Problem:** Understand the Gymnasium API and its core abstractions (environments, spaces, observations, rewards) by running a random baseline agent on two standard environments: FrozenLake-v1 and Taxi-v3.

- **Method:** Built a reusable framework with an abstract Agent base class, an episodic runner that collects per-episode returns, lengths, and success indicators, and an environment inspector that programmatically extracts space descriptions and transition tables. Ran a RandomAgent for 1,000 episodes on each environment.

- **Design choice:** Reused the propose-review-consent-implement workflow from Part One, adapting the bandit framework for episodic environments with terminal states and variable-length episodes. Added an `env_inspector` module to expose MDP structure programmatically.

- **Key result:** The random agent establishes a clear performance baseline on both environments. FrozenLake's slippery dynamics make even intentional navigation unreliable, while Taxi's compound action structure (movement plus pickup/dropoff) and penalty system make random success effectively impossible.

- **Insight:** Programmatic environment introspection (e.g., extracting transition tables) is more reliable than relying on descriptions. Verifying FrozenLake's slip probabilities directly revealed that the intended action only executes one-third of the time, a detail that was initially understated.

- **Challenge:** Adapting the fixed-horizon bandit runner to handle episodic environments required rethinking the simulation loop to account for terminal states, variable episode lengths, and success criteria.

- **Connection:** The random baseline and environment introspection tools built here provide the foundation for Week 2, where policy iteration and value iteration will be applied to these same environments, and the performance improvements over random action selection will quantify the value of learning.